# Limpieza, análisis y estructuración de datos con Pandas (Instacart)

**Actividad:** GA1-220501094-01-AA1-EV06

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno contiene los laboratorios guiados y los retos de las lecciones 2, 3 y 4. Los ejemplos de la explicación están en `pandas_examples.ipynb`.

## Preparación

Carga de los datos crudos y limpieza de la explicación de la Lección 2. Las tablas `orders`, `products` y `order_prods` limpias se usan en los laboratorios y retos siguientes.

In [31]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')

orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

print(pd.__version__)
print(orders.shape, products.shape, order_prods.shape, aisles.shape, departments.shape)
# Esperado: (478967, 6) (49694, 4) (4545007, 4) (134, 2) (21, 2)

3.0.6
(478967, 6) (49694, 4) (4545007, 4) (134, 2) (21, 2)


In [32]:
# Limpieza manual de la explicación de la Lección 2
orders = orders.drop_duplicates().reset_index(drop=True)
orders = orders.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                        'order_dow': 'int8', 'order_hour_of_day': 'int8',
                        'days_since_prior_order': 'Int16'})

order_prods = order_prods.astype({'order_id': 'int32', 'product_id': 'int32',
                                  'add_to_cart_order': 'Int16', 'reordered': 'int8'})

products['product_name'] = (products['product_name'].fillna('Unknown')
                            .str.replace('\xa0', ' ', regex=False)
                            .str.replace(r'\s+', ' ', regex=True)
                            .str.strip().str.title())

assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Tablas limpias listas')

Tablas limpias listas


# Lección 2: Limpieza y estandarización

## Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Objetivo: construir una secuencia de pasos de limpieza que parte de los archivos crudos, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso.

### Paso 1: recargar los datos crudos

In [33]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)
# Esperado: (478967, 6) (49694, 4) (4545007, 4)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: una función que anota el estado de la tabla

In [34]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por función

In [35]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                        .str.replace('\xa0', ' ', regex=False)
                        .str.replace(r'\s+', ' ', regex=True)
                        .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                    'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

Si `registro` tiene filas repetidas, vuelve a ejecutar desde la celda `registro = []`.

In [36]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [37]:
informe = pd.DataFrame(registro)
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**

- Quitar duplicados eliminó 15 filas de `orders` y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- Corregir los tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- Los 1 258 nulos de `product_name` se rellenaron con `'Unknown'`.
- Los 836 nulos de `order_prods` se conservan a propósito (ver Reto 2, punto 3).

El informe se guarda en `data/processed/cleaning_report.csv`.

### Paso 6: validar el resultado

In [38]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))
# Esperado: Validación superada / True True

Validación superada
True True


### Conclusiones del laboratorio

- El pipeline produce exactamente las mismas tablas que la limpieza manual, pero se puede volver a ejecutar con una sola instrucción cuando lleguen datos nuevos.
- El informe responde con cifras a la pregunta "¿qué le hiciste a los datos?".
- La función `validar` convierte las reglas de calidad en una prueba que se ejecuta cada vez.

## Reto 2

### 1. Duplicados y valores de `order_prods`

¿Hay filas duplicadas exactas? ¿Se repite algún par `order_id` + `product_id`? ¿`reordered` solo contiene 0 y 1?

In [39]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:',
      order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))
print('¿Solo 0 y 1?', order_prods['reordered'].isin([0, 1]).all())

Filas duplicadas exactas: 0
Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]
¿Solo 0 y 1? True


**Interpretación:** `order_prods` no tiene filas duplicadas exactas ni pares `order_id` + `product_id` repetidos, así que esa combinación funciona como clave compuesta: cada producto aparece una sola vez por pedido. `reordered` solo contiene 0 y 1. Esta tabla no necesita limpieza de duplicados ni de valores.

### 2. Texto en `aisles` y `departments`

¿Tienen nombres duplicados o con espacios al inicio o al final? ¿Necesitan limpieza de texto?

In [40]:
for nombre, df, col in [('aisles', aisles, 'aisle'),
                       ('departments', departments, 'department')]:
    texto = df[col]
    print(nombre)
    print('  nulos:', texto.isna().sum())
    print('  duplicados exactos:', texto.duplicated().sum())
    print('  duplicados al ignorar mayúsculas y espacios:',
          texto.str.lower().str.strip().duplicated().sum())
    print('  con espacios al inicio o al final:', (texto != texto.str.strip()).sum())
    print('  con espacios dobles:', texto.str.contains('  ', regex=False).sum())

aisles
  nulos: 0
  duplicados exactos: 0
  duplicados al ignorar mayúsculas y espacios: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0
departments
  nulos: 0
  duplicados exactos: 0
  duplicados al ignorar mayúsculas y espacios: 0
  con espacios al inicio o al final: 0
  con espacios dobles: 0


**Interpretación:** `aisles` y `departments` no tienen nulos, nombres duplicados (ni siquiera al ignorar mayúsculas y espacios) ni espacios sobrantes. No necesitan limpieza de texto. Lo único a tener en cuenta es el valor centinela `missing` de `departments` (punto 7).

### 3. Los 836 nulos de `add_to_cart_order`

¿En cuántos pedidos ocurren? ¿Cuántos productos tiene, como mínimo, cada uno de esos pedidos? ¿Qué los explica?

In [41]:
nulos_carrito = order_prods[order_prods['add_to_cart_order'].isna()]
pedidos_con_nulos = nulos_carrito['order_id'].unique()
print('Filas con nulo:', len(nulos_carrito))
print('Pedidos afectados:', len(pedidos_con_nulos))

productos_por_pedido = order_prods.groupby('order_id').size()
print('Mínimo de productos en esos pedidos:', productos_por_pedido.loc[pedidos_con_nulos].min())
print('Posición máxima registrada en add_to_cart_order:', order_prods['add_to_cart_order'].max())

Filas con nulo: 836
Pedidos afectados: 70
Mínimo de productos en esos pedidos: 65
Posición máxima registrada en add_to_cart_order: 64


**Interpretación:** los 836 nulos están en solo 70 pedidos, y cada uno de esos pedidos tiene al menos 65 productos. Como la posición máxima registrada en `add_to_cart_order` es 64, el sistema solo guarda la posición del carrito hasta el producto 64: a partir del 65 queda vacía. No es un error de los datos sino un límite del registro, por eso se **conservan** los nulos (tipo `Int16`). Rellenarlos inventaría posiciones.

### 4. Nombres que cambiaron con la estandarización

Cuenta cuántos nombres (no nulos) de `products.csv` cambiaron con la limpieza.

In [42]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')

crudos = products_crudo['product_name']
limpios = products['product_name']
no_nulos = crudos.notna()

cambiaron = (crudos[no_nulos] != limpios[no_nulos])
print('Nombres no nulos:', no_nulos.sum())
print('Nombres que cambiaron:', cambiaron.sum())
pd.DataFrame({'antes': crudos[no_nulos][cambiaron], 'despues': limpios[no_nulos][cambiaron]}).head()

Nombres no nulos: 48436
Nombres que cambiaron: 6321


,antes,despues
17,Pizza for One Suprema Frozen Pizza,Pizza For One Suprema Frozen Pizza
23,Tri-Vi-Sol® Vitamins A-C-and D Supplement Drop...,Tri-Vi-Sol® Vitamins A-C-And D Supplement Drop...
29,Three Cheese Ziti- Marinara with Meatballs,Three Cheese Ziti- Marinara With Meatballs
35,Traditional Lasagna with Meat Sauce Savory Ita...,Traditional Lasagna With Meat Sauce Savory Ita...
41,Biotin 1000 mcg,Biotin 1000 Mcg


**Interpretación:** de 48 436 nombres no nulos, 6 321 cambiaron con la estandarización (cerca del 13 %). La mayoría de los cambios son de mayúsculas por `.str.title()` (por ejemplo, `for` → `For`) y una parte son espacios dobles o invisibles. Una limitación: `.title()` también cambia siglas y unidades (`mcg` → `Mcg`), lo cual no afecta el análisis porque se agrupa por `product_id`.

### 5. Día de la semana como categoría ordenada

Series con el nombre del día de cada pedido, convertida a categoría ordenada de domingo a sábado. Comparación de memoria.

In [43]:
dias = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']

nombre_dia = orders['order_dow'].map(dict(enumerate(dias)))
dia_categoria = pd.Series(pd.Categorical(nombre_dia, categories=dias, ordered=True),
                          index=nombre_dia.index)

mem_texto = nombre_dia.memory_usage(deep=True) / 1e6
mem_categoria = dia_categoria.memory_usage(deep=True) / 1e6
print(f'Texto: {mem_texto:.2f} MB')
print(f'Categoría: {mem_categoria:.2f} MB')
print(f'La categoría ocupa {mem_categoria / mem_texto:.1%} del texto')
print(dia_categoria.value_counts().sort_index())

Texto: 7.14 MB
Categoría: 0.48 MB
La categoría ocupa 6.7% del texto
domingo      84090
lunes        82185
martes       65833
miércoles    60897
jueves       59810
viernes      63488
sábado       62649
Name: count, dtype: int64


**Interpretación:** el texto ocupa 7,14 MB y la categoría 0,48 MB, apenas el 6,7 % (unas 15 veces menos), porque la categoría guarda cada uno de los 7 nombres una sola vez y en cada fila solo un código. Al ser **ordenada**, `sort_index()` muestra los días de domingo a sábado y no en orden alfabético. El domingo es el día con más pedidos (84 090) y el jueves el de menos (59 810).

### 6. Productos `'Unknown'` en los pedidos

¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Los eliminarías?

In [44]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]

print('Filas con productos Unknown:', len(filas_unknown))
print('Pedidos en los que aparecen:', filas_unknown['order_id'].nunique())
print(f'Porcentaje de las filas de order_prods: {len(filas_unknown) / len(order_prods):.2%}')

Filas con productos Unknown: 9547
Pedidos en los que aparecen: 8254
Porcentaje de las filas de order_prods: 0.21%


**Interpretación:** hay 9 547 filas de `order_prods` con productos `Unknown`, repartidas en 8 254 pedidos. Son solo el 0,21 % de las filas, pero son compras reales.

**¿Los eliminarías?** No. Son compras reales: el producto existe y tiene `product_id`, aunque falte su nombre en el catálogo. Eliminarlos reduciría el tamaño real de esos pedidos y alteraría métricas como la tasa de recompra o los artículos por pedido. Es mejor conservarlos y, si un análisis lo necesita, excluirlos solo en ese análisis.

### 7. `'missing'` como nulo real en una copia de `departments`

In [45]:
deptos = departments.copy()
deptos['department'] = deptos['department'].replace('missing', pd.NA)

print('Nulos en la copia:', deptos['department'].isna().sum())
print('Nulos en la original:', departments['department'].isna().sum())
deptos[deptos['department'].isna()]

Nulos en la copia: 1
Nulos en la original: 0


,department_id,department
20,21,NaN


**¿Por qué no hacerlo en la tabla original antes de agrupar?** Porque `groupby` descarta por defecto las claves nulas (`dropna=True`): los productos de ese departamento desaparecerían en silencio de los resúmenes, mientras que con el texto `'missing'` siguen visibles como un grupo más.

<!-- leccion-3 -->
## Lección 3: Filtrado, ordenamiento y agrupación

### Laboratorio guiado 3: perfil de hábitos de compra

**Encargo:** el área de mercadeo quiere diseñar campañas según qué tan seguido compra cada cliente, y necesita saber **a qué hora compra cada tipo de cliente**.

Plan:

1. Contar los pedidos de cada cliente.
2. Clasificar a los clientes en tres segmentos.
3. Clasificar los pedidos en franjas horarias.
4. Cruzar segmentos y franjas.

In [46]:
# Comprobación: tablas limpias de la Lección 2
import numpy as np

print(orders.shape, order_prods.shape)
assert orders.shape == (478952, 6), 'orders no está limpia: ejecuta el cuaderno desde el principio'
# Esperado: (478952, 6) (4545007, 4)

(478952, 6) (4545007, 4)


#### Paso 1: pedidos por cliente

Pregunta: "número de pedidos **por** cliente".

In [47]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()
# Esperado: 157437 clientes, mediana (50%) de 2 pedidos y máximo de 28

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

#### Paso 2: segmentar clientes con `pd.cut`

- **ocasional:** 1 pedido
- **regular:** 2 a 4 pedidos
- **frecuente:** 5 o más pedidos

In [48]:
segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))
# Esperado:
# ocasional  55357  35.2 %
# regular    71553  45.4 %
# frecuente  30527  19.4 %

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


#### Paso 3: llevar el segmento a cada pedido con `map`

In [49]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()
# Esperado: columna segmento con ocasional, regular o frecuente en cada fila

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


#### Paso 4: resumen por segmento

In [50]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento
# Esperado:
# segmento   clientes  pedidos  espera_mediana
# ocasional  55357     55357    16
# regular    71553     191649   11
# frecuente  30527     231946   6

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace casi la mitad de los pedidos (231 946 de 478 952) y vuelve cada 6 días. Los ocasionales tienen espera porque su único pedido en estos datos no es necesariamente el primero que hicieron en la tienda.

#### Paso 5: franjas horarias y tabla cruzada

Los límites empiezan en –1 para que la hora 0 entre en la madrugada, porque el límite izquierdo no se incluye.

Además del material, el perfil se guarda en `data/processed/segment_hour_profile.csv`.

In [51]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil.to_csv('../data/processed/segment_hour_profile.csv')
perfil
# Esperado (% por fila):
# segmento   madrugada  mañana  tarde  noche
# ocasional  2.1        30.0    49.4   18.5
# regular    1.9        31.6    48.4   18.1
# frecuente  1.7        35.2    45.9   17.2

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


**Interpretación:** todos los segmentos compran sobre todo en la tarde, pero los frecuentes compran más en la mañana (35,2 % frente a 30,0 % de los ocasionales). Una campaña de fidelización debería llegar por la mañana.

#### Paso 6: las combinaciones de día y franja más activas

In [52]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))
# Esperado: domingo (0) tarde 42340, lunes (1) tarde 37432, martes (2) tarde 30751

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

#### Paso 7: dejar `orders` como estaba

Se eliminan las columnas auxiliares para no arrastrarlas a la Lección 4. Si esta celda se ejecuta dos veces da `KeyError`: en ese caso, ejecuta el cuaderno desde el principio.

In [53]:
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)
# Esperado: (478952, 6)

(478952, 6)


#### Conclusiones clave

- Una pregunta de negocio compleja se resuelve en pasos: agrupar, categorizar, devolver al detalle y cruzar.
- `pd.cut` convierte números en categorías con límites explícitos.
- `map` lleva un resultado calculado por grupo a cada fila del detalle.
- `crosstab` con `normalize` permite comparar grupos de distinto tamaño.

### Reto 3

En cada punto se escribe primero la pregunta como "métrica por grupo" o como condición, y después el código. En `order_dow`, 0 es domingo y 6 es sábado.

#### 1. Las tres horas con más pedidos los domingos

**Pregunta:** número de pedidos **por** hora, con la condición `order_dow == 0`.

In [54]:
domingos = orders[orders['order_dow'] == 0]
top_horas_domingo = domingos.groupby('order_hour_of_day')['order_id'].count().nlargest(3)
top_horas_domingo

order_hour_of_day
15    7585
14    7533
13    7483
Name: order_id, dtype: int64

**Interpretación:** los domingos el pico está a primera hora de la tarde: 15 h (7 585 pedidos), 14 h (7 533) y 13 h (7 483). Es distinto del pico general de la semana, que es a las 10 a. m. (40 578 pedidos). Las tres horas tienen cifras muy parecidas, así que el domingo la actividad alta ocupa toda la franja de 13 a 15 h.

#### 2. Horas con la mayor y la menor espera media

**Pregunta:** media de `days_since_prior_order` **por** hora. `mean` ignora los nulos de los primeros pedidos.

In [55]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean().round(2)
print('Mayor espera media:', espera_por_hora.idxmax(), 'h ->', espera_por_hora.max(), 'días')
print('Menor espera media:', espera_por_hora.idxmin(), 'h ->', espera_por_hora.min(), 'días')

Mayor espera media: 0 h -> 12.52 días
Menor espera media: 22 h -> 10.62 días


**Interpretación:** la mayor espera media es a las 0 h (12,52 días) y la menor a las 22 h (10,62 días). La diferencia es de casi 2 días: los pedidos de medianoche vienen de clientes que tardaron más en volver, y los de las 22 h de clientes que vuelven antes. Es una diferencia moderada; la hora sola no separa con claridad a los clientes.

#### 3. Clientes con 10 pedidos o más

**Pregunta:** número de pedidos **por** cliente, con la condición `>= 10`.

In [56]:
pedidos_cliente = orders.groupby('user_id')['order_id'].count()
clientes_10 = (pedidos_cliente >= 10).sum()
print('Clientes con 10 pedidos o más:', clientes_10)
print(f'Porcentaje de los clientes: {clientes_10 / len(pedidos_cliente):.2%}')

Clientes con 10 pedidos o más: 6327
Porcentaje de los clientes: 4.02%


**Interpretación:** 6 327 clientes tienen 10 pedidos o más, el 4,02 % de los 157 437 clientes. Es un grupo pequeño dentro de los 30 527 clientes frecuentes del laboratorio, y es el núcleo más fiel de la tienda.

#### 4. Pedidos hechos exactamente 7 días después del anterior

**Condición:** `days_since_prior_order == 7`. El porcentaje se calcula sobre el total de pedidos (478 952), que incluye los primeros pedidos sin espera.

In [57]:
es_7_dias = orders['days_since_prior_order'].eq(7)
pedidos_7 = es_7_dias.sum()
print('Pedidos a los 7 días:', pedidos_7)
print(f'Porcentaje del total: {pedidos_7 / len(orders):.2%}')

Pedidos a los 7 días: 44577
Porcentaje del total: 9.31%


**Interpretación:** 44 577 pedidos se hicieron exactamente 7 días después del anterior, el 9,31 % del total. Si se excluyen los 28 817 primeros pedidos, que no tienen espera, la proporción sube a cerca del 9,9 %. Casi 1 de cada 10 pedidos sigue un ciclo semanal, lo que sugiere un hábito de compra de cada semana.

#### 5. Pedidos con 20 productos o más y el pedido más grande

**Pregunta:** número de productos **por** pedido en `order_prods`, con la condición `>= 20`.

In [58]:
productos_por_pedido = order_prods.groupby('order_id')['product_id'].count()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(),
      'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420
Pedido más grande: 61355 con 127 productos


**Interpretación:** 49 420 pedidos tienen 20 productos o más, cerca del 11 % de los 450 046 pedidos que tienen productos en `order_prods`. El pedido más grande es el 61355, con 127 productos: un caso extremo, muy por encima de un pedido típico.

#### 6. Tasa de recompra en las posiciones 1 a 5 del carrito

**Pregunta:** media de `reordered` **por** posición en el carrito (`add_to_cart_order`), con la condición de posición entre 1 y 5.

In [59]:
primeras_posiciones = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
tasa_recompra = (primeras_posiciones
                 .groupby('add_to_cart_order')['reordered']
                 .mean()
                 .mul(100).round(1))
tasa_recompra   # en porcentaje

add_to_cart_order
1    67.9
2    67.7
3    65.9
4    63.8
5    61.7
Name: reordered, dtype: float64

**Patrón observado:** la tasa de recompra baja a medida que avanza la posición en el carrito: 67,9 % en la posición 1, 67,7 % en la 2, 65,9 % en la 3, 63,8 % en la 4 y 61,7 % en la 5. Los clientes empiezan agregando los productos que ya compran siempre y dejan para después los productos nuevos.

#### 7. Mediana de espera: fin de semana frente al resto de la semana

**Pregunta:** mediana de `days_since_prior_order` **por** tipo de día. Se agrupa directamente sobre la máscara `isin([0, 6])`.

In [60]:
es_fin_semana = orders['order_dow'].isin([0, 6])
mediana_tipo_dia = (orders
                    .groupby(es_fin_semana)['days_since_prior_order']
                    .median()
                    .rename(index={True: 'fin de semana', False: 'entre semana'}))
mediana_tipo_dia

order_dow
entre semana     7.0
fin de semana    8.0
Name: days_since_prior_order, dtype: Float64

**Interpretación:** la mediana de espera es de 8 días en los pedidos de fin de semana y de 7 días en los de entre semana. Quienes compran el sábado o el domingo tardan en la mediana un día más en volver. La diferencia es pequeña, pero coincide con la espera media por día, que es más alta el domingo (11,8) y el sábado (11,4).